# 04) Train with W&B

## Setup

Loads `data/train.csv`/`data/columns.json` fresh (same convention as every other notebook here),
logs in to W&B (credentials already cached in `~/.netrc` -- no key ever passed here), and rebuilds
the exact 10-fold expanding-window CV split from `03)`: walk-forward, `MIN_TRAIN_DAYS=7` warm-up,
`DELTA_DAYS=1` step, no lookahead.

The loaded DataFrame is `train_df`, not `train` -- `train` is reserved for the per-model function
each W&B sweep calls.

In [1]:
import json
import os

import numpy as np
import pandas as pd
import wandb
from scipy.stats import spearmanr
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

train_df = pd.read_csv("data/train.csv", parse_dates=["date"])
with open("data/columns.json") as f:
    columns = json.load(f)

TARGET_COL = columns["target_cols"][0]
FINAL_FEATURE_COLS = columns["final_feature_cols"]

os.environ["WANDB_SILENT"] = "true"  # suppress wandb's own per-run console noise
wandb.login()

print(f"train_df: {train_df.shape[0]} rows, target = {TARGET_COL!r}")
print(f"{len(FINAL_FEATURE_COLS)} final features:", FINAL_FEATURE_COLS)

train_df: 8443 rows, target = 'target_bps_vs_mid'
4 final features: ['near_price_vs_mid_bps', 'urgency', 'ref_price_vs_mid_bps', 'far_price_vs_mid_bps']


In [2]:
unique_dates = np.sort(train_df["date"].unique())
MIN_TRAIN_DAYS = 7
DELTA_DAYS = 1

folds = []
for fold_idx, eval_start in enumerate(range(MIN_TRAIN_DAYS, len(unique_dates), DELTA_DAYS)):
    fit_dates = unique_dates[:eval_start]
    eval_dates = unique_dates[eval_start:eval_start + DELTA_DAYS]
    assert fit_dates.max() < eval_dates.min(), "eval dates must all fall after fit dates"

    fit_mask = train_df["date"].isin(fit_dates)
    eval_mask = train_df["date"].isin(eval_dates)
    folds.append((fit_mask, eval_mask))

    print(f"fold {fold_idx}: fit {fit_mask.sum():>4} rows ({len(fit_dates)} days, "
          f"{pd.Timestamp(fit_dates.min()).date()} to {pd.Timestamp(fit_dates.max()).date()}) | "
          f"eval {eval_mask.sum():>4} rows ({len(eval_dates)} days, "
          f"{pd.Timestamp(eval_dates.min()).date()} to {pd.Timestamp(eval_dates.max()).date()})")

fold 0: fit 3476 rows (7 days, 2025-03-03 to 2025-03-11) | eval  496 rows (1 days, 2025-03-12 to 2025-03-12)
fold 1: fit 3972 rows (8 days, 2025-03-03 to 2025-03-12) | eval  497 rows (1 days, 2025-03-13 to 2025-03-13)
fold 2: fit 4469 rows (9 days, 2025-03-03 to 2025-03-13) | eval  497 rows (1 days, 2025-03-14 to 2025-03-14)
fold 3: fit 4966 rows (10 days, 2025-03-03 to 2025-03-14) | eval  497 rows (1 days, 2025-03-17 to 2025-03-17)
fold 4: fit 5463 rows (11 days, 2025-03-03 to 2025-03-17) | eval  496 rows (1 days, 2025-03-18 to 2025-03-18)
fold 5: fit 5959 rows (12 days, 2025-03-03 to 2025-03-18) | eval  497 rows (1 days, 2025-03-19 to 2025-03-19)
fold 6: fit 6456 rows (13 days, 2025-03-03 to 2025-03-19) | eval  496 rows (1 days, 2025-03-20 to 2025-03-20)
fold 7: fit 6952 rows (14 days, 2025-03-03 to 2025-03-20) | eval  497 rows (1 days, 2025-03-21 to 2025-03-21)
fold 8: fit 7449 rows (15 days, 2025-03-03 to 2025-03-21) | eval  497 rows (1 days, 2025-03-24 to 2025-03-24)
fold 9: fit 7

## Shared sklearn sweep runner

One `train()`-function factory reused by every sklearn model's sweep (`HuberRegressor`, `Lasso`,
`GradientBoostingRegressor`, `RandomForestRegressor`): builds `impute -> scale -> model`, fits/evals
across all 10 folds, logs the full metric set. `random_state` is fixed to `0` automatically for
models that accept it (not swept -- a determinism knob, not a tunable choice).

In [ ]:
def make_sklearn_train_fn(estimator_class):
    def train():
        wandb.init()
        config = dict(wandb.config)
        if "random_state" in estimator_class().get_params():
            config["random_state"] = 0
        estimator = estimator_class(**config)

        model = Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("scale", StandardScaler()),
            ("model", estimator),
        ])

        fit_mses, eval_mses = [], []
        fit_r2s, eval_r2s = [], []
        fit_ics, eval_ics = [], []
        for fit_mask, eval_mask in folds:
            X_fit = train_df.loc[fit_mask, FINAL_FEATURE_COLS]
            y_fit = train_df.loc[fit_mask, TARGET_COL]
            X_eval = train_df.loc[eval_mask, FINAL_FEATURE_COLS]
            y_eval = train_df.loc[eval_mask, TARGET_COL]

            model.fit(X_fit, y_fit)
            pred_fit = model.predict(X_fit)
            pred_eval = model.predict(X_eval)

            fit_mses.append(mean_squared_error(y_fit, pred_fit))
            eval_mses.append(mean_squared_error(y_eval, pred_eval))
            fit_r2s.append(r2_score(y_fit, pred_fit))
            eval_r2s.append(r2_score(y_eval, pred_eval))
            fit_ics.append(spearmanr(y_fit, pred_fit).statistic)
            eval_ic = spearmanr(y_eval, pred_eval).statistic
            if np.isnan(eval_ic):
                # constant predictions (e.g. Lasso alpha large enough to zero every
                # coefficient) make Spearman IC undefined -- clip to the worst possible
                # IC so the Bayes optimizer sees a well-defined bad value, not NaN.
                eval_ic = -1.0
            eval_ics.append(eval_ic)

        metrics = {
            "fit_mse": np.mean(fit_mses),
            "eval_mse_mean": np.mean(eval_mses),
            "eval_mse_std": np.std(eval_mses),
            "fit_r2": np.mean(fit_r2s),
            "eval_r2": np.mean(eval_r2s),
            "fit_ic": np.mean(fit_ics),
            "eval_ic": np.mean(eval_ics),
        }
        wandb.log(metrics)
        print(f"  [{wandb.run.name}] eval_ic={metrics['eval_ic']:.4f}  eval_mse={metrics['eval_mse_mean']:.2f}")
        wandb.finish()
    return train

print("make_sklearn_train_fn ready")

## HuberRegressor sweep (best linear)

`epsilon` controls where Huber's loss switches from quadratic to linear (robustness to outliers);
`alpha` is the L2 penalty strength. 100-trial Bayesian search, optimizing held-out IC.

In [ ]:
from sklearn.linear_model import HuberRegressor

huber_sweep_config = {
    "method": "bayes",
    "metric": {"name": "eval_ic", "goal": "maximize"},
    "parameters": {
        "epsilon": {"distribution": "log_uniform_values", "min": 1.1, "max": 3.0},
        "alpha": {"distribution": "log_uniform_values", "min": 1e-5, "max": 1e1},
    },
}

huber_sweep_id = wandb.sweep(huber_sweep_config, project="close-auction-model-tuning")
print("huber_sweep_id:", huber_sweep_id)
wandb.agent(huber_sweep_id, function=make_sklearn_train_fn(HuberRegressor), project="close-auction-model-tuning", count=100)

_api = wandb.Api()
_best = _api.sweep(f"{_api.default_entity}/close-auction-model-tuning/{huber_sweep_id}").best_run()
print(f"best: {_best.name}  eval_ic={_best.summary_metrics['eval_ic']:.4f}  "
      f"eval_mse_mean={_best.summary_metrics['eval_mse_mean']:.2f}  config={dict(_best.config)}")

## Lasso sweep (baseline)

Plain L1-penalized linear regression -- the explicit floor to compare the other four models
against. Only `alpha` to tune. 100-trial Bayesian search, optimizing held-out IC.

In [ ]:
from sklearn.linear_model import Lasso

lasso_sweep_config = {
    "method": "bayes",
    "metric": {"name": "eval_ic", "goal": "maximize"},
    "parameters": {
        "alpha": {"distribution": "log_uniform_values", "min": 1e-5, "max": 1e4},
    },
}

lasso_sweep_id = wandb.sweep(lasso_sweep_config, project="close-auction-model-tuning")
print("lasso_sweep_id:", lasso_sweep_id)
wandb.agent(lasso_sweep_id, function=make_sklearn_train_fn(Lasso), project="close-auction-model-tuning", count=100)

_api = wandb.Api()
_best = _api.sweep(f"{_api.default_entity}/close-auction-model-tuning/{lasso_sweep_id}").best_run()
print(f"best: {_best.name}  eval_ic={_best.summary_metrics['eval_ic']:.4f}  "
      f"eval_mse_mean={_best.summary_metrics['eval_mse_mean']:.2f}  config={dict(_best.config)}")

Create sweep with ID: m2dpwcxx
Sweep URL: https://wandb.ai/ablancoprada-massachusetts-institute-of-technology/close-auction-model-tuning/sweeps/m2dpwcxx
  [devoted-sweep-1] eval_ic=0.8208  eval_mse=10.0773
  [misunderstood-sweep-2] eval_ic=nan  eval_mse=72.0647
  [lively-sweep-3] eval_ic=0.8207  eval_mse=10.0785
  [comic-sweep-4] eval_ic=0.8207  eval_mse=10.0781
  [tough-sweep-5] eval_ic=0.8206  eval_mse=12.1086
  [autumn-sweep-6] eval_ic=0.8207  eval_mse=10.0776
  [serene-sweep-7] eval_ic=0.8207  eval_mse=10.0787
  [prime-sweep-8] eval_ic=0.8207  eval_mse=10.0773
  [toasty-sweep-9] eval_ic=0.8207  eval_mse=10.0777
  [fast-sweep-10] eval_ic=0.8207  eval_mse=10.0781
  [fine-sweep-11] eval_ic=0.8207  eval_mse=10.0781
  [fallen-sweep-12] eval_ic=0.8207  eval_mse=10.0773
  [mild-sweep-13] eval_ic=0.8208  eval_mse=10.0773
  [lemon-sweep-14] eval_ic=0.8207  eval_mse=10.0773
  [jumping-sweep-15] eval_ic=0.8207  eval_mse=10.0781
  [peachy-sweep-16] eval_ic=0.8207  eval_mse=10.0773
  [icy-sweep

## GradientBoostingRegressor sweep (best boosted tree)

`min_samples_leaf`'s range goes wide and regularization-leaning on purpose -- `03)`'s shortlist run
showed the untuned tree models overfitting badly, so the sweep needs room to find configs that
generalize, not just variations near sklearn's overfit-prone defaults. 100-trial Bayesian search,
optimizing held-out IC.

In [6]:
from sklearn.ensemble import GradientBoostingRegressor

gb_sweep_config = {
    "method": "bayes",
    "metric": {"name": "eval_ic", "goal": "maximize"},
    "parameters": {
        "n_estimators": {"distribution": "int_uniform", "min": 50, "max": 400},
        "max_depth": {"distribution": "int_uniform", "min": 2, "max": 6},
        "learning_rate": {"distribution": "log_uniform_values", "min": 1e-3, "max": 3e-1},
        "subsample": {"distribution": "uniform", "min": 0.5, "max": 1.0},
        "min_samples_leaf": {"distribution": "int_uniform", "min": 1, "max": 50},
    },
}

gb_sweep_id = wandb.sweep(gb_sweep_config, project="close-auction-model-tuning")
print("gb_sweep_id:", gb_sweep_id)
wandb.agent(gb_sweep_id, function=make_sklearn_train_fn(GradientBoostingRegressor), project="close-auction-model-tuning", count=100)

_api = wandb.Api()
_best = _api.sweep(f"{_api.default_entity}/close-auction-model-tuning/{gb_sweep_id}").best_run()
print(f"best: {_best.name}  eval_ic={_best.summary_metrics['eval_ic']:.4f}  "
      f"eval_mse_mean={_best.summary_metrics['eval_mse_mean']:.2f}  config={dict(_best.config)}")

Create sweep with ID: 97pw7tca
Sweep URL: https://wandb.ai/ablancoprada-massachusetts-institute-of-technology/close-auction-model-tuning/sweeps/97pw7tca
  [feasible-sweep-1] eval_ic=0.8184  eval_mse=24.6609
  [wise-sweep-2] eval_ic=0.8180  eval_mse=24.2989
  [laced-sweep-3] eval_ic=0.8217  eval_mse=36.5363
  [distinctive-sweep-4] eval_ic=0.8201  eval_mse=47.5317
  [toasty-sweep-5] eval_ic=0.8219  eval_mse=30.7854
  [twilight-sweep-6] eval_ic=0.8219  eval_mse=40.1687
  [solar-sweep-7] eval_ic=0.8213  eval_mse=57.1514
  [polished-sweep-8] eval_ic=0.8212  eval_mse=50.5378
  [devout-sweep-9] eval_ic=0.8223  eval_mse=35.8114
  [quiet-sweep-10] eval_ic=0.8211  eval_mse=49.0043
  [charmed-sweep-11] eval_ic=0.8216  eval_mse=64.1094
  [chocolate-sweep-12] eval_ic=0.8220  eval_mse=36.9136
  [bright-sweep-13] eval_ic=0.8220  eval_mse=13.1347
  [spring-sweep-14] eval_ic=0.8218  eval_mse=23.4903
  [fancy-sweep-15] eval_ic=0.8228  eval_mse=12.2114
  [fast-sweep-16] eval_ic=0.8216  eval_mse=27.7187
 

## RandomForestRegressor sweep (best bagged tree)

Same regularization-leaning `min_samples_leaf` range as GradientBoosting above, for the same
reason -- `03)`'s shortlist run showed this exact model overfitting badly at sklearn defaults
(fit MSE 3.0 vs. eval MSE 15.4, a 5x gap). 100-trial Bayesian search, optimizing held-out IC.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf_sweep_config = {
    "method": "bayes",
    "metric": {"name": "eval_ic", "goal": "maximize"},
    "parameters": {
        "n_estimators": {"distribution": "int_uniform", "min": 50, "max": 400},
        "max_depth": {"distribution": "int_uniform", "min": 2, "max": 15},
        "min_samples_leaf": {"distribution": "int_uniform", "min": 1, "max": 50},
        "max_features": {"values": ["sqrt", "log2", 0.5, 1.0]},
    },
}

rf_sweep_id = wandb.sweep(rf_sweep_config, project="close-auction-model-tuning")
print("rf_sweep_id:", rf_sweep_id)
wandb.agent(rf_sweep_id, function=make_sklearn_train_fn(RandomForestRegressor), project="close-auction-model-tuning", count=100)

_api = wandb.Api()
_best = _api.sweep(f"{_api.default_entity}/close-auction-model-tuning/{rf_sweep_id}").best_run()
print(f"best: {_best.name}  eval_ic={_best.summary_metrics['eval_ic']:.4f}  "
      f"eval_mse_mean={_best.summary_metrics['eval_mse_mean']:.2f}  config={dict(_best.config)}")

## MLP: PyTorch architecture + per-fold training loop

sklearn's `MLPRegressor` only supports `{identity, logistic, tanh, relu}` activations and has no
clean hook for per-epoch validation MSE across arbitrary CV folds -- both needed here (leaky ReLU,
and a real train/eval-vs-epoch curve). Built as a small `torch.nn` feedforward net instead. The
final layer is always a plain `nn.Linear` with no activation after it -- linear output for a
continuous regression target.

`train_one_fold_torch` runs one fold: Adam optimizer, MSE loss, `torch.manual_seed(0)` for
reproducibility. Early stopping uses an inner train/val split carved out of the fit data (10%
held out, patience 20, max 200 epochs, keeps the best-epoch weights) -- the outer eval fold is
never touched until the final prediction, so early-stopping never peeks at held-out data.
`log_epochs=True` (used for fold 9 only, the largest) logs a full per-epoch curve to W&B *and*
prints a compact progress line every 20 epochs, so training is visible directly in the notebook
output too, not just on the W&B dashboard.

In [ ]:
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split

ACTIVATIONS = {"relu": nn.ReLU, "leaky_relu": nn.LeakyReLU}


class MLPNet(nn.Module):
    def __init__(self, input_dim, hidden_layer_sizes, activation):
        super().__init__()
        act_cls = ACTIVATIONS[activation]
        layers = []
        prev_size = input_dim
        for size in hidden_layer_sizes:
            layers.append(nn.Linear(prev_size, size))
            layers.append(act_cls())
            prev_size = size
        layers.append(nn.Linear(prev_size, 1))  # linear output, no activation
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)


def train_one_fold_torch(X_fit, y_fit, X_eval, y_eval, hidden_layer_sizes, activation,
                          alpha, learning_rate_init, max_epochs=200, patience=20,
                          log_epochs=False, print_every=20):
    torch.manual_seed(0)  # determinism knob, matching the sklearn side's random_state=0
    device = torch.device("cpu")

    # Early stopping must never look at X_eval/y_eval (the outer CV fold) -- that would be
    # model selection on held-out data. Carve an inner train/val split out of the fit data
    # instead, mirroring sklearn.MLPRegressor(early_stopping=True, validation_fraction=...).
    X_train, X_val, y_train, y_val = train_test_split(
        X_fit, y_fit, test_size=0.1, random_state=0
    )
    X_train_t = torch.tensor(X_train, dtype=torch.float32, device=device)
    y_train_t = torch.tensor(y_train, dtype=torch.float32, device=device)
    X_val_t = torch.tensor(X_val, dtype=torch.float32, device=device)
    y_val_t = torch.tensor(y_val, dtype=torch.float32, device=device)
    X_fit_t = torch.tensor(X_fit, dtype=torch.float32, device=device)
    X_eval_t = torch.tensor(X_eval, dtype=torch.float32, device=device)

    model = MLPNet(X_fit.shape[1], tuple(hidden_layer_sizes), activation).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate_init, weight_decay=alpha)
    loss_fn = nn.MSELoss()

    best_val_mse = float("inf")
    best_state = None
    epochs_without_improvement = 0
    final_epoch = 0

    for epoch in range(max_epochs):
        final_epoch = epoch
        model.train()
        optimizer.zero_grad()
        loss = loss_fn(model(X_train_t), y_train_t)
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            train_mse = loss_fn(model(X_train_t), y_train_t).item()
            val_mse = loss_fn(model(X_val_t), y_val_t).item()

        if log_epochs:
            if wandb.run is not None:
                wandb.log({"epoch": epoch, "fold9_train_mse": train_mse, "fold9_val_mse": val_mse})
            if epoch % print_every == 0:
                print(f"    epoch {epoch:>3}: train_mse={train_mse:.3f}  val_mse={val_mse:.3f}")

        if val_mse < best_val_mse:
            best_val_mse = val_mse
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                break

    if log_epochs:
        print(f"    stopped at epoch {final_epoch} (best val_mse={best_val_mse:.3f})")

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        pred_fit = model(X_fit_t).numpy()
        pred_eval = model(X_eval_t).numpy()
    return pred_fit, pred_eval

print("MLPNet and train_one_fold_torch ready")

**Sanity check** -- run one fold (fold 9, the largest) directly, no W&B sweep, just confirming it
trains and the loss actually goes down before wiring this into a 100-trial sweep. `log_epochs=True`
here too, so you can watch the epoch progress live for this one check, same as it'll look during
the real sweep.

In [13]:
_fit_mask, _eval_mask = folds[9]
_X_fit_raw = train_df.loc[_fit_mask, FINAL_FEATURE_COLS].to_numpy()
_y_fit = train_df.loc[_fit_mask, TARGET_COL].to_numpy()
_X_eval_raw = train_df.loc[_eval_mask, FINAL_FEATURE_COLS].to_numpy()
_y_eval = train_df.loc[_eval_mask, TARGET_COL].to_numpy()

_imputer = SimpleImputer(strategy="median").fit(_X_fit_raw)
_X_fit_imp = _imputer.transform(_X_fit_raw)
_X_eval_imp = _imputer.transform(_X_eval_raw)
_scaler = StandardScaler().fit(_X_fit_imp)
_X_fit_scaled = _scaler.transform(_X_fit_imp)
_X_eval_scaled = _scaler.transform(_X_eval_imp)

_pred_fit, _pred_eval = train_one_fold_torch(
    _X_fit_scaled, _y_fit, _X_eval_scaled, _y_eval,
    hidden_layer_sizes=(64, 32), activation="relu",
    alpha=1e-3, learning_rate_init=1e-2, log_epochs=True,
)
print("sanity-check fold 9 -- fit MSE:", mean_squared_error(_y_fit, _pred_fit),
      " eval MSE:", mean_squared_error(_y_eval, _pred_eval))

    epoch   0: train_mse=73.697  eval_mse=57.593
    epoch  20: train_mse=20.000  eval_mse=17.546
    epoch  40: train_mse=13.266  eval_mse=10.165
    epoch  60: train_mse=11.995  eval_mse=9.006
    epoch  80: train_mse=11.856  eval_mse=9.087
    stopped at epoch 80 (best eval_mse=9.006)
sanity-check fold 9 -- fit MSE: 11.99457170140093  eval MSE: 9.006104632387736


## MLP sweep (best neural net)

Wires `MLPNet`/`train_one_fold_torch` into a real W&B sweep. Same impute/scale-per-fold discipline
as the sklearn models (fit-fold statistics only, applied to both sides). Fold 9 -- the largest --
gets its per-epoch train/val MSE logged to W&B *and* printed every 20 epochs on every trial; the
other 9 folds still contribute their final-epoch (best early-stopped) metrics to the aggregate.
Each trial also prints a one-line summary when it finishes, same compact format as the sklearn
sweeps above. 100-trial Bayesian search, optimizing held-out IC.

In [14]:
def mlp_train():
    wandb.init()
    config = wandb.config

    fit_mses, eval_mses = [], []
    fit_r2s, eval_r2s = [], []
    fit_ics, eval_ics = [], []
    for fold_idx, (fit_mask, eval_mask) in enumerate(folds):
        X_fit_raw = train_df.loc[fit_mask, FINAL_FEATURE_COLS].to_numpy()
        y_fit = train_df.loc[fit_mask, TARGET_COL].to_numpy()
        X_eval_raw = train_df.loc[eval_mask, FINAL_FEATURE_COLS].to_numpy()
        y_eval = train_df.loc[eval_mask, TARGET_COL].to_numpy()

        imputer = SimpleImputer(strategy="median").fit(X_fit_raw)
        X_fit_imp = imputer.transform(X_fit_raw)
        X_eval_imp = imputer.transform(X_eval_raw)
        scaler = StandardScaler().fit(X_fit_imp)
        X_fit_scaled = scaler.transform(X_fit_imp)
        X_eval_scaled = scaler.transform(X_eval_imp)

        is_fold9 = (fold_idx == 9)
        if is_fold9:
            print(f"  [{wandb.run.name}] fold 9:")

        pred_fit, pred_eval = train_one_fold_torch(
            X_fit_scaled, y_fit, X_eval_scaled, y_eval,
            hidden_layer_sizes=config["hidden_layer_sizes"],
            activation=config["activation"],
            alpha=config["alpha"],
            learning_rate_init=config["learning_rate_init"],
            log_epochs=is_fold9,
        )

        fit_mses.append(mean_squared_error(y_fit, pred_fit))
        eval_mses.append(mean_squared_error(y_eval, pred_eval))
        fit_r2s.append(r2_score(y_fit, pred_fit))
        eval_r2s.append(r2_score(y_eval, pred_eval))
        fit_ics.append(spearmanr(y_fit, pred_fit).statistic)
        eval_ics.append(spearmanr(y_eval, pred_eval).statistic)

    metrics = {
        "fit_mse": np.mean(fit_mses),
        "eval_mse_mean": np.mean(eval_mses),
        "eval_mse_std": np.std(eval_mses),
        "fit_r2": np.mean(fit_r2s),
        "eval_r2": np.mean(eval_r2s),
        "fit_ic": np.mean(fit_ics),
        "eval_ic": np.mean(eval_ics),
    }
    wandb.log(metrics)
    print(f"  [{wandb.run.name}] eval_ic={metrics['eval_ic']:.4f}  eval_mse={metrics['eval_mse_mean']:.2f}")
    wandb.finish()

mlp_sweep_config = {
    "method": "bayes",
    "metric": {"name": "eval_ic", "goal": "maximize"},
    "parameters": {
        "hidden_layer_sizes": {"values": [[16], [32], [64], [128], [32, 16], [64, 32],
                                            [128, 64], [64, 32, 16], [128, 64, 32], [100, 50]]},
        "activation": {"values": ["relu", "leaky_relu"]},
        "alpha": {"distribution": "log_uniform_values", "min": 1e-5, "max": 1e0},
        "learning_rate_init": {"distribution": "log_uniform_values", "min": 1e-4, "max": 1e-1},
    },
}

mlp_sweep_id = wandb.sweep(mlp_sweep_config, project="close-auction-model-tuning")
print("mlp_sweep_id:", mlp_sweep_id)
wandb.agent(mlp_sweep_id, function=mlp_train, project="close-auction-model-tuning", count=100)

_api = wandb.Api()
_best = _api.sweep(f"{_api.default_entity}/close-auction-model-tuning/{mlp_sweep_id}").best_run()
print(f"best: {_best.name}  eval_ic={_best.summary_metrics['eval_ic']:.4f}  "
      f"eval_mse_mean={_best.summary_metrics['eval_mse_mean']:.2f}  config={dict(_best.config)}")

Create sweep with ID: e26neauh
Sweep URL: https://wandb.ai/ablancoprada-massachusetts-institute-of-technology/close-auction-model-tuning/sweeps/e26neauh
mlp_sweep_id: e26neauh
  [electric-sweep-1] fold 9:
    epoch   0: train_mse=76.814  eval_mse=60.381
    epoch  20: train_mse=68.825  eval_mse=52.525
    epoch  40: train_mse=46.504  eval_mse=31.015
    epoch  60: train_mse=19.788  eval_mse=11.719
    stopped at epoch 79 (best eval_mse=11.694)
  [electric-sweep-1] eval_ic=0.8176  eval_mse=11.10
  [dainty-sweep-2] fold 9:
    epoch   0: train_mse=74.034  eval_mse=57.942
    epoch  20: train_mse=71.540  eval_mse=55.518
    epoch  40: train_mse=69.082  eval_mse=53.136
    epoch  60: train_mse=66.609  eval_mse=50.753
    epoch  80: train_mse=64.071  eval_mse=48.337
    epoch 100: train_mse=61.451  eval_mse=45.884
    epoch 120: train_mse=58.743  eval_mse=43.376
    epoch 140: train_mse=55.951  eval_mse=40.808
    epoch 160: train_mse=53.087  eval_mse=38.200
    epoch 180: train_mse=50.174 